# CLV Segmentation, XGBoost, and Campaign ROI

This notebook operationalizes CLV predictions into actionable business outputs:

1. **4-tier customer segmentation** based on predicted CLV and P(alive)
2. **Segment profiles**: size, avg CLV, predicted revenue, recommended actions
3. **XGBoost CLV regressor**: adds engagement signals the probabilistic model ignores
4. **Campaign ROI allocation table**: budget optimization by segment

**Outputs:**
- `models/xgb_clv_model.pkl` — XGBoost CLV regressor
- `data/processed/clv_final.csv` — customers with segments and all CLV features

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb

# Paths
SCORED_PATH = "../data/processed/clv_scored.csv"
FINAL_PATH  = "../data/processed/clv_final.csv"
XGB_PATH    = "../models/xgb_clv_model.pkl"

print("Libraries loaded.")

## 1. Load Data

In [ ]:
df = pd.read_csv(SCORED_PATH)
print(f"Loaded {len(df):,} customers × {df.shape[1]} features")
print(f"\nCLV range: ${df['clv_12m'].min():.2f} — ${df['clv_12m'].max():.2f}")
print(f"p_alive range: {df['p_alive'].min():.4f} — {df['p_alive'].max():.4f}")

## 2. 4-Tier Customer Segmentation

| Segment | Definition | Recommended Action |
|---------|------------|--------------------|
| **High Value** | Top 20% CLV | No discounts — protect margin |
| **At-Risk** | p_alive < 0.3 (any CLV band) | Win-back campaign |
| **Growing** | Middle 40% CLV + p_alive ≥ 0.3 | Personalized offers |
| **Low Value** | Bottom 40% CLV + p_alive ≥ 0.3 | Email-only, minimal spend |

**Note:** At-Risk is checked across all CLV bands and takes priority over Growing/Low Value.

In [ ]:
# Define CLV thresholds
clv_top20_threshold    = df['clv_12m'].quantile(0.80)  # top 20%
clv_bottom40_threshold = df['clv_12m'].quantile(0.40)  # bottom 40%

print(f"CLV thresholds:")
print(f"  Top 20% (High Value):   CLV > ${clv_top20_threshold:.2f}")
print(f"  Middle 40% (Growing):   ${clv_bottom40_threshold:.2f} < CLV ≤ ${clv_top20_threshold:.2f}")
print(f"  Bottom 40% (Low Value): CLV ≤ ${clv_bottom40_threshold:.2f}")
print(f"  At-Risk:                p_alive < 0.3 (overrides other segments)")

def assign_segment(row):
    if row['clv_12m'] > clv_top20_threshold:
        return 'High Value'
    elif row['p_alive'] < 0.3:
        return 'At-Risk'
    elif row['clv_12m'] > clv_bottom40_threshold:
        return 'Growing'
    else:
        return 'Low Value'

df['segment'] = df.apply(assign_segment, axis=1)

segment_counts = df['segment'].value_counts()
print(f"\nSegment distribution:")
for seg, cnt in segment_counts.items():
    print(f"  {seg:15s}: {cnt:,} customers ({cnt/len(df):.1%})")

## 3. Segment Profiles

In [ ]:
# Segment profile table
seg_profile = df.groupby('segment').agg(
    n_customers       = ('user_id', 'count'),
    avg_clv_12m       = ('clv_12m', 'mean'),
    median_clv_12m    = ('clv_12m', 'median'),
    total_pred_revenue = ('clv_12m', 'sum'),
    avg_p_alive       = ('p_alive', 'mean'),
    avg_frequency     = ('frequency', 'mean'),
    avg_monetary      = ('monetary_value', 'mean'),
).round(2)

# Add segment share of total predicted revenue
seg_profile['revenue_share'] = (seg_profile['total_pred_revenue'] / seg_profile['total_pred_revenue'].sum() * 100).round(1)

# Reorder rows
seg_order = ['High Value', 'Growing', 'At-Risk', 'Low Value']
seg_profile = seg_profile.reindex([s for s in seg_order if s in seg_profile.index])

print("=== Segment Profile ===")
print(seg_profile.to_string())

In [ ]:
# Segment visualization
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

colors = {'High Value': '#2196F3', 'Growing': '#4CAF50', 'At-Risk': '#FF5722', 'Low Value': '#9E9E9E'}
segs = [s for s in seg_order if s in seg_profile.index]

# Customer count by segment
counts = [seg_profile.loc[s, 'n_customers'] for s in segs]
bar_colors = [colors[s] for s in segs]
axes[0].bar(segs, counts, color=bar_colors, edgecolor='white')
axes[0].set_title('Customer Count by Segment')
axes[0].set_ylabel('Customers')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
for i, v in enumerate(counts):
    axes[0].text(i, v + max(counts)*0.01, f'{v:,}', ha='center', fontsize=9)

# Predicted revenue by segment
revenues = [seg_profile.loc[s, 'total_pred_revenue'] for s in segs]
axes[1].bar(segs, revenues, color=bar_colors, edgecolor='white')
axes[1].set_title('Predicted 12-Month Revenue by Segment')
axes[1].set_ylabel('Predicted Revenue ($)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
for i, (v, s) in enumerate(zip(revenues, segs)):
    share = seg_profile.loc[s, 'revenue_share']
    axes[1].text(i, v + max(revenues)*0.01, f'{share:.1f}%', ha='center', fontsize=9)

plt.suptitle('CLV Segment Overview', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. XGBoost CLV Regressor

The XGBoost model complements BG/NBD by incorporating **engagement signals** (sessions, events, cart activity) that the probabilistic model ignores. It provides a second CLV estimate for comparison and feature importance insights.

**Target:** `log1p(clv_12m)` — log-transformed to handle the heavy-tailed distribution.

In [ ]:
# Feature engineering for XGBoost
NUMERIC_FEATURES = [
    # BG/NBD inputs
    'frequency', 'recency', 'T', 'monetary_value',
    # Transaction context
    'total_orders', 'avg_order_value', 'days_since_last_order',
    # Demographics
    'customer_tenure_days', 'age',
    # Engagement
    'total_sessions', 'total_events', 'days_since_last_visit',
    'avg_events_per_session', 'cart_events', 'product_view_events',
]
CATEGORICAL_FEATURES = ['gender', 'traffic_source', 'country']

# Encode categoricals
df_model = df.copy()
label_encoders = {}
for col in CATEGORICAL_FEATURES:
    le = LabelEncoder()
    df_model[col + '_enc'] = le.fit_transform(df_model[col].astype(str))
    label_encoders[col] = le

FEATURE_COLS = NUMERIC_FEATURES + [c + '_enc' for c in CATEGORICAL_FEATURES]
X = df_model[FEATURE_COLS]
y = np.log1p(df_model['clv_12m'])

print(f"Features: {len(FEATURE_COLS)}")
print(f"Target:   log1p(clv_12m), range [{y.min():.3f}, {y.max():.3f}]")

In [ ]:
# Train / test split (temporal: top 20% by T as test — most recent customers)
# Using random split here for simplicity; temporal split used in validation notebook
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

# Fit XGBoost
xgb_model = xgb.XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbosity=0,
)
xgb_model.fit(X_train, y_train)

# Evaluate
y_pred_log = xgb_model.predict(X_test)
y_pred     = np.expm1(y_pred_log)
y_true     = np.expm1(y_test)

r2  = r2_score(y_test, y_pred_log)
mae = mean_absolute_error(y_true, y_pred)

print("=== XGBoost CLV Regressor ===")
print(f"R² (log scale):  {r2:.4f}")
print(f"MAE ($ scale):   ${mae:.2f} per customer")

In [ ]:
# Feature importance
importance = pd.Series(xgb_model.feature_importances_, index=FEATURE_COLS)
top_features = importance.nlargest(15).sort_values()

fig, ax = plt.subplots(figsize=(9, 6))
top_features.plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('XGBoost Feature Importance (Top 15)')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.show()

In [ ]:
# Add XGBoost CLV estimate to dataframe
df['xgb_clv_12m'] = np.expm1(xgb_model.predict(X[FEATURE_COLS]))

# Compare BG/NBD vs XGBoost CLV at segment level
comparison = df.groupby('segment')[['clv_12m', 'xgb_clv_12m']].mean().round(2)
comparison.columns = ['BG/NBD CLV (avg)', 'XGBoost CLV (avg)']
print("=== CLV Model Comparison by Segment ===")
print(comparison.to_string())

## 5. Campaign ROI Allocation Table

Budget-per-customer recommendations based on segment economics:

| Segment | Budget/Customer | Conversion Rate | Expected Margin | Rationale |
|---------|----------------|-----------------|-----------------|----------|
| High Value | $0 (organic) | 40% | ~CLV | Retain naturally; discounts destroy margin |
| Growing | $15 | 25% | CLV × 0.25 − $15 | Personalized offer justified by growth potential |
| At-Risk | $10 | 15% | CLV × 0.15 − $10 | Win-back campaign; lower success rate |
| Low Value | $2 (email) | 5% | CLV × 0.05 − $2 | Email only; minimum investment |

In [ ]:
# Campaign economics
campaign_params = {
    'High Value': {'budget_per_customer': 0,  'conversion_rate': 0.40, 'action': 'VIP loyalty — no campaign'},
    'Growing':    {'budget_per_customer': 15, 'conversion_rate': 0.25, 'action': 'Personalized offer'},
    'At-Risk':    {'budget_per_customer': 10, 'conversion_rate': 0.15, 'action': 'Win-back campaign'},
    'Low Value':  {'budget_per_customer': 2,  'conversion_rate': 0.05, 'action': 'Email only'},
}

roi_rows = []
for seg in seg_order:
    if seg not in seg_profile.index:
        continue
    params   = campaign_params[seg]
    n_cust   = seg_profile.loc[seg, 'n_customers']
    avg_clv  = seg_profile.loc[seg, 'avg_clv_12m']
    budget   = params['budget_per_customer']
    conv     = params['conversion_rate']

    total_cost    = budget * n_cust
    expected_rev  = avg_clv * conv * n_cust
    net_roi       = expected_rev - total_cost
    roi_pct       = (net_roi / total_cost * 100) if total_cost > 0 else float('inf')

    roi_rows.append({
        'Segment':            seg,
        'Customers':          n_cust,
        'Budget/Customer':    f'${budget}',
        'Conversion Rate':    f'{conv:.0%}',
        'Avg CLV':            f'${avg_clv:,.2f}',
        'Total Campaign Cost': f'${total_cost:,.0f}',
        'Expected Revenue':   f'${expected_rev:,.0f}',
        'Net ROI':            f'${net_roi:,.0f}',
        'ROI %':              f'{roi_pct:.0f}%' if total_cost > 0 else 'N/A (no spend)',
        'Action':             params['action'],
    })

roi_df = pd.DataFrame(roi_rows)
print("=== Campaign ROI Allocation Table ===")
print(roi_df.to_string(index=False))

## 6. Save Final Outputs

In [ ]:
# Save XGBoost model
os.makedirs('../models', exist_ok=True)
joblib.dump(xgb_model, XGB_PATH)
print(f"✓ XGBoost model saved to {XGB_PATH}")

# Save final CLV dataset with segments
os.makedirs('../data/processed', exist_ok=True)
df.to_csv(FINAL_PATH, index=False)
print(f"✓ Final CLV data saved to {FINAL_PATH}")
print(f"  Shape: {df.shape[0]:,} customers × {df.shape[1]} columns")
print(f"  Columns include: segment, clv_12m, p_alive, xgb_clv_12m")

In [ ]:
# Final summary
print("=" * 55)
print("CLV SEGMENTATION SUMMARY")
print("=" * 55)
print(f"Total customers:      {len(df):,}")
print(f"Total predicted CLV:  ${df['clv_12m'].sum():,.0f}")
print(f"Median CLV:           ${df['clv_12m'].median():.2f}")
print()
for seg in seg_order:
    if seg in seg_profile.index:
        n   = int(seg_profile.loc[seg, 'n_customers'])
        rev = seg_profile.loc[seg, 'total_pred_revenue']
        pct = seg_profile.loc[seg, 'revenue_share']
        print(f"  {seg:15s}: {n:,} customers | ${rev:>10,.0f} predicted ({pct:.1f}%)")
print("=" * 55)
print("\nNext step: streamlit run src/app.py")

---

**Next:** [src/app.py](../src/app.py) — Launch the CLV dashboard with `streamlit run src/app.py`